In [3]:
import pandas as pd
from silver_staging_utils import connect_to_postgres, read_table
import numpy as np
import json
import re
import unicodedata

MAPPINGS_FILE = '/workspaces/tesis-ivan-gennaro/scripts/silver_staging/final_category_mappings.json'

# Productos

In [4]:
query = '''
SELECT
    *
FROM silver.products
WHERE
    snapshot_date IN (
        SELECT DISTINCT snapshot_date FROM silver.products ORDER BY snapshot_date DESC LIMIT 10   
    )
'''
print(query)

conn = connect_to_postgres()
if conn:
    df = read_table(query, conn)
    conn.close()


SELECT
    *
FROM silver.products
WHERE
    snapshot_date IN (
        SELECT DISTINCT snapshot_date FROM silver.products ORDER BY snapshot_date DESC LIMIT 10   
    )

✅ Conectado a PostgreSQL


/workspaces/tesis-ivan-gennaro/scripts/silver_staging/silver_staging_utils.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 107285 entries, 0 to 107284
Data columns (total 12 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   snapshot_date    107285 non-null  object        
 1   supermarket      107285 non-null  object        
 2   product_id       107285 non-null  object        
 3   product_name     107285 non-null  object        
 4   brand            40707 non-null   object        
 5   price            107285 non-null  object        
 6   unit_of_measure  66578 non-null   object        
 7   is_on_promotion  66578 non-null   object        
 8   promotion_price  71172 non-null   object        
 9   category_slug    107285 non-null  object        
 10  ingestion_time   107285 non-null  datetime64[ns]
 11  created_at       107285 non-null  datetime64[ns]
dtypes: datetime64[ns](2), object(10)
memory usage: 9.8+ MB


In [6]:
df['snapshot_date'].unique()

array([datetime.date(2025, 11, 12), datetime.date(2025, 11, 16),
       datetime.date(2025, 11, 13), datetime.date(2025, 11, 14),
       datetime.date(2025, 11, 11), datetime.date(2025, 11, 10),
       datetime.date(2025, 11, 15), datetime.date(2025, 11, 9),
       datetime.date(2025, 11, 8), datetime.date(2025, 11, 7)],
      dtype=object)

In [7]:
df['supermarket'].unique()

array(['biggie', 'real'], dtype=object)

## Final Price creation

##### Price problem with Casa Rica

In [8]:
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace(r"[^\d,\.]", "", regex=True)
    .str.replace(".", "", regex=False) 
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

##### Coalesce to build final price

In [9]:
df.loc[df['promotion_price'] == '0', 'promotion_price'] = None

In [10]:
df["final_price"] = (
    df["promotion_price"]
        .combine_first(df['price'])
)

# Categoria

In [11]:
query = '''
SELECT
    *
FROM silver.categories
WHERE
    snapshot_date IN (
        SELECT MAX(snapshot_date) FROM silver.categories  
    )
'''
print(query)

conn = connect_to_postgres()
if conn:
    df = read_table(query, conn)
    conn.close()


SELECT
    *
FROM silver.categories
WHERE
    snapshot_date IN (
        SELECT MAX(snapshot_date) FROM silver.categories  
    )

✅ Conectado a PostgreSQL


/workspaces/tesis-ivan-gennaro/scripts/silver_staging/silver_staging_utils.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1427 entries, 0 to 1426
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   snapshot_date       1427 non-null   object        
 1   supermarket         1427 non-null   object        
 2   category_lvl1_name  1427 non-null   object        
 3   category_lvl2_name  1382 non-null   object        
 4   category_lvl3_name  1057 non-null   object        
 5   category_lvl1_id    45 non-null     object        
 6   category_lvl2_id    0 non-null      object        
 7   category_lvl3_id    0 non-null      object        
 8   category_lvl1_slug  45 non-null     object        
 9   category_lvl2_slug  325 non-null    object        
 10  category_lvl3_slug  1057 non-null   object        
 11  created_at          1427 non-null   datetime64[ns]
dtypes: datetime64[ns](1), object(11)
memory usage: 133.9+ KB


In [13]:
df['snapshot_date'].unique()

array([datetime.date(2025, 11, 16)], dtype=object)

In [14]:
df['supermarket'].unique()

array(['biggie', 'real', 's6', 'stock', 'casa_rica'], dtype=object)

### category_final_slug
Creacion de un slug homogeneo para todos los supermercados, a utilizar para la surrogate key

In [15]:
mask_real = df['supermarket'] == 'real'

df.loc[mask_real, 'real_lvl1_clean'] = (
    df.loc[mask_real, 'category_lvl1_slug']
        .str.split('/', n=1)
        .str[-1]
)


In [16]:
df['category_slug_final'] = (
    np.where(df['supermarket'] == 'biggie', df['category_lvl1_slug'],
    np.where(df['supermarket'] == 'casa rica', df['category_lvl2_slug'],
    np.where(df['supermarket'].isin(['stock', 'super seis']), df['category_lvl3_slug'],
    np.where(df['supermarket'] == 'real', df['real_lvl1_clean'],
             None))))
)

### category_final_name

Análisis

In [17]:
query = '''
SELECT
    supermarket,
	category_lvl1_name
FROM silver.categories
'''
print(query)

conn = connect_to_postgres()
if conn:
    df = read_table(query, conn)
    conn.close()


SELECT
    supermarket,
	category_lvl1_name
FROM silver.categories

✅ Conectado a PostgreSQL


/workspaces/tesis-ivan-gennaro/scripts/silver_staging/silver_staging_utils.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [18]:
df.head()

,supermarket,category_lvl1_name
0,biggie,Alimentos Especiales
1,biggie,Almacén
2,biggie,Asado
3,biggie,Bebes
4,biggie,Bebidas con Alcohol


In [19]:
pivot = (
    df
    .assign(value=True)          # Marcamos presencia
    .pivot_table(
        index='category_lvl1_name',
        columns='supermarket',
        values='value',
        aggfunc='any',           # Si existe al menos 1 → True
        fill_value=False
    )
)

pivot_int = pivot.astype(int)


In [20]:
pivot_int.to_csv("/workspaces/tesis-ivan-gennaro/scripts/silver_staging/category_final_name_analisis.csv")

Aplicacion

In [21]:
def normalize_text(s):
    """
    Normaliza una cadena para matching:
      - pasa a minúsculas
      - quita acentos
      - reemplaza caracteres no alfanuméricos por espacio
      - reduce espacios múltiples a uno
      - strip()
    """
    if s is None:
        return None
    s = str(s).strip().lower()
    # quitar acentos
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    # deja solo letras, números y espacios
    s = re.sub(r"[^a-z0-9\s]+", " ", s)
    # colapsar espacios
    s = re.sub(r"\s+", " ", s).strip()
    return s


In [22]:
df['category_clean'] = df['category_lvl1_name'].apply(normalize_text)


In [23]:
# Leer mappings desde JSON
with open(MAPPINGS_FILE, 'r', encoding='utf-8') as f:
    mappings = json.load(f)

inv_maps = {sup: {} for sup in ["biggie", "casa_rica", "real", "s6", "stock"]}

for final, mapping in mappings.items():
    for sup, cats in mapping.items():
        if cats is None:
            continue
        
        # Normalizar: convertir string -> lista de strings
        if isinstance(cats, str):
            cats = [cats]

        # Normalizar cada valor
        cats_norm = [normalize_text(c) for c in cats]

        # Guardar mapping categoria_super → categoria_final
        for c in cats_norm:
            inv_maps[sup][c] = final   # final NO normalizado, mejor para presentación


In [24]:
# 3) Mapear la categoría final
df['category_final'] = df.apply(
    lambda row: inv_maps[row['supermarket']].get(row['category_clean']),
    axis=1
)

In [25]:
df.drop_duplicates().to_csv("/workspaces/tesis-ivan-gennaro/scripts/silver_staging/category_final_name_mapped.csv")

In [26]:
df[df["category_final"].isna()][["supermarket", "category_lvl1_name", "category_clean"]].drop_duplicates()

,supermarket,category_lvl1_name,category_clean
70,biggie,None,None
37704,s6,Ferretería,ferreteria
37724,s6,Fiambrería,fiambreria
37734,s6,Frescos,frescos
37751,s6,Hogar y bazar,hogar y bazar
37790,s6,Juguetes y librería,juguetes y libreria
37849,s6,Mascotas,mascotas
37871,s6,Pastas,pastas
37927,s6,Repostería,reposteria


In [27]:
df["category_final"].drop_duplicates()

0                 Almacén
2                   Otros
3                   Bebes
4                 Bebidas
6                  Carnes
8              Congelados
9       Fiambres y Quesos
10      Frutas y Verduras
12               Farmacia
13                Lácteos
14               Librería
15               Limpieza
16               Mascotas
17              Panadería
19                  Bazar
70                   None
2662       Pastas Frescas
5531    Electrodomésticos
Name: category_final, dtype: object